In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import random
from collections import Counter
import re
import time
from torch.cuda.amp import autocast, GradScaler

# ============================================================
# FIXES APPLIED:
# 1. Augmentation ONLY on train split (no leakage into dev)
# 2. Dev set uses ORIGINAL data only (no augmented samples)
# 3. Stratified train/dev split to preserve class distribution
# 4. BiLSTM uses attention pooling instead of last hidden state
# 5. Label smoothing re-added alongside class weights
# 6. Focal loss option for hard imbalanced classes
# 7. Tokenizer fit ONLY on train texts (no dev leakage)
# 8. Lazy tokenization in Dataset (avoids memory/index bugs)
# ============================================================

# ------------------ 1. DATA AUGMENTATION ------------------
def synonym_replacement(text, synonym_dict, n=2):
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word in synonym_dict]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = synonym_dict.get(random_word, [])
        if synonyms:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return ' '.join(new_words)

synonym_dict = {
    'നല്ല': ['മികച്ച', 'ഉത്തമ', 'ശ്രേഷ്ഠ', 'പ്രശംസനീയ', 'സന്തോഷകരം'],
    'സന്തോഷം': ['ആനന്ദം', 'ഹർഷം', 'തൃപ്തി'],
    'സ്നേഹം': ['പ്രേമം', 'അനുരാഗം', 'സൗഹൃദം'],
    'മനോഹരം': ['ആകർഷകമായ', 'സുന്ദരം', 'അലങ്കാരികം'],
    'മോശം': ['പിന്നോക്ക', 'താഴ്ന്ന', 'നിന്ദ്യ', 'അസഹ്യമായ', 'അവമതിപ്പുള്ള'],
    'വെറുപ്പ്': ['ദ്വേഷം', 'അസൂയ', 'അനിഷ്ടം'],
    'വേദന': ['നൊവു', 'കഷ്ടം', 'തകർപ്പ്'],
    'തട്ടിപ്പുകാരൻ': ['വഞ്ചകൻ', 'കള്ളൻ', 'മോഷ്ടാവ്'],
    'മണ്ടൻ': ['അബുദ്ധിമാൻ', 'അവിവേകി', 'വിവേകശൂന്യൻ'],
    'ഭീകരൻ': ['തീവ്രവാദി', 'അക്രമി', 'ഭീഷണിയുള്ള'],
    'കള്ളൻ': ['മോഷ്ടാവ്', 'വഞ്ചകൻ', 'തട്ടിപ്പുകാരൻ'],
    'വീട്': ['ഗൃഹം', 'വാസസ്ഥലം', 'ഇല്ലം'],
    'കൂട്ടുകാരൻ': ['സുഹൃത്ത്', 'മിത്രം', 'സംഗതി'],
}

# ------------------ 2. TEXT CLEANING ------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ------------------ 3. FOCAL LOSS (handles imbalance better) ------------------
class FocalLoss(nn.Module):
    """
    Focal Loss down-weights easy examples so the model focuses on hard ones.
    Better than pure class-weighted CE for severe imbalance.
    """
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

# ------------------ 4. LAZY-TOKENIZED DATASET ------------------
# FIX: Tokenization happens at __getitem__ time, not upfront in bulk.
# This eliminates index mismatch bugs when data is augmented/shuffled,
# and avoids storing thousands of BERT tensors in RAM simultaneously.
class MalayalamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer1, tokenizer2, tokenizer_cnn, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer1 = tokenizer1
        self.tokenizer2 = tokenizer2
        self.tokenizer_cnn = tokenizer_cnn
        self.max_len = max_len

        # Pre-compute CNN/RNN sequences (lightweight, safe to do upfront)
        self.rnn_inputs = [
            pad_sequences(
                tokenizer_cnn.texts_to_sequences([text]), maxlen=max_len
            )[0]
            for text in texts
        ]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # BERT tokenization at access time — no bulk memory / index bugs
        enc1 = self.tokenizer1(
            text,
            return_tensors='pt',
            truncation=True,
            padding='max_length',
            max_length=self.max_len
        )
        enc2 = self.tokenizer2(
            text,
            return_tensors='pt',
            truncation=True,
            padding='max_length',
            max_length=self.max_len
        )

        return {
            'rnn_input':       torch.tensor(self.rnn_inputs[idx], dtype=torch.long),
            'input_ids_1':     enc1['input_ids'].squeeze(0),
            'attention_mask_1':enc1['attention_mask'].squeeze(0),
            'input_ids_2':     enc2['input_ids'].squeeze(0),
            'attention_mask_2':enc2['attention_mask'].squeeze(0),
            'label':           torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ------------------ 5. LOAD & PREPROCESS DATA ------------------
file_path_1 = '/content/mal_full_offensive_dev.csv'
file_path_2 = '/content/mal_full_offensive_train.csv'

df1 = pd.read_csv(file_path_1)
df2 = pd.read_csv(file_path_2)
df = pd.concat([df1, df2], ignore_index=True)
df.columns = ['text', 'class']
df['text'] = df['text'].apply(clean_text)
df['label'] = df['class'].map({
    'Not_offensive': 0,
    'Offensive_Targeted_Insult_Individual': 1,
    'Offensive_Targeted_Insult_Group': 2,
    'Offensive_Untargetede': 3,
    'not-malayalam': 4
})
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df = df[df['label'].isin([0, 1, 2, 3, 4])]

# ---------------------------------------------------------------
# FIX 1: Stratified split BEFORE augmentation
# Dev set stays clean (original distribution, no augmented samples)
# ---------------------------------------------------------------
X_raw = df['text'].tolist()
y_raw = df['label'].tolist()

X_train_raw, X_dev, y_train_raw, y_dev = train_test_split(
    X_raw, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw          # <-- preserves class ratios in both splits
)

# ---------------------------------------------------------------
# FIX 2: Augmentation applied ONLY to training portion
# ---------------------------------------------------------------
class_counts = Counter(y_train_raw)
max_count = max(class_counts.values())

X_train, y_train = [], []
for text, label in zip(X_train_raw, y_train_raw):
    X_train.append(text)
    y_train.append(label)
    if label in [1, 2, 3, 4]:
        # Cap augmentation at 3x to avoid overwhelming with near-duplicates
        num_aug = min(int(max_count / class_counts[label]) - 1, 3)
        for _ in range(num_aug):
            X_train.append(synonym_replacement(text, synonym_dict))
            y_train.append(label)

print(f"Train size after augmentation: {len(X_train)}")
print(f"Dev size (original only):      {len(X_dev)}")
print(f"Train class distribution: {Counter(y_train)}")
print(f"Dev class distribution:   {Counter(y_dev)}")

# ---------------------------------------------------------------
# FIX 3: CNN tokenizer fit ONLY on train texts (no dev leakage)
# ---------------------------------------------------------------
max_words = 15000
max_len   = 128

tokenizer_cnn = Tokenizer(num_words=max_words)
tokenizer_cnn.fit_on_texts(X_train)   # <-- was fit on ALL data before

# ------------------ 6. BERT TOKENIZERS ------------------
tokenizer1 = AutoTokenizer.from_pretrained("xlm-roberta-base")
tokenizer2 = AutoTokenizer.from_pretrained("google/muril-base-cased")

# Build datasets
train_dataset = MalayalamDataset(X_train, y_train, tokenizer1, tokenizer2, tokenizer_cnn, max_len)
dev_dataset   = MalayalamDataset(X_dev,   y_dev,   tokenizer1, tokenizer2, tokenizer_cnn, max_len)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Compute class weights from TRAINING data only
num_classes   = 5
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_counts  = Counter(y_train)
total_train   = len(y_train)
class_weights = torch.tensor(
    [total_train / (num_classes * train_counts[i]) for i in range(num_classes)],
    dtype=torch.float
).to(device)

# ------------------ 7. ATTENTION POOLING FOR BiLSTM ------------------
class AttentionPooling(nn.Module):
    """
    FIX: Instead of taking only the last hidden state (which loses most
    of the sequence), we learn a weighted average over all time steps.
    This gives the BiLSTM branch a much richer representation.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden_dim)
        scores = self.attention(lstm_out).squeeze(-1)          # (batch, seq_len)
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)  # (batch, seq_len, 1)
        pooled = (lstm_out * weights).sum(dim=1)               # (batch, hidden_dim)
        return pooled

# ------------------ 8. MULTI-HEAD ATTENTION FUSION ------------------
class MultiHeadFusion(nn.Module):
    def __init__(self, hidden_dim=768, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=heads, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        residual = x.mean(dim=1)
        x, _ = self.attn(x, x, x)
        # Residual connection for stability
        return self.norm(x.mean(dim=1) + residual)

# ------------------ 9. FUSION MODEL ------------------
class FusionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_classes=5):
        super().__init__()

        # --- BiLSTM branch ---
        self.embedding  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bilstm     = nn.LSTM(
            embed_dim, 256,
            batch_first=True,
            bidirectional=True,
            num_layers=2,
            dropout=0.3        # inter-layer dropout
        )
        self.attn_pool  = AttentionPooling(512)   # 256*2 bidirectional
        self.rnn_proj   = nn.Linear(512, 768)

        # --- Transformer branches ---
        self.bert1 = AutoModel.from_pretrained("xlm-roberta-base")
        self.bert2 = AutoModel.from_pretrained("google/muril-base-cased")

        # Freeze only bottom 6 layers (fine-tune top 6)
        for param in self.bert1.encoder.layer[:6].parameters():
            param.requires_grad = False
        for param in self.bert2.encoder.layer[:6].parameters():
            param.requires_grad = False

        # --- Fusion & Classifier ---
        self.fusion     = MultiHeadFusion(768, heads=8)
        self.dropout    = nn.Dropout(0.4)
        self.ffn = nn.Sequential(
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, input_ids_1, attention_mask_1,
                input_ids_2, attention_mask_2, rnn_input):

        # BiLSTM branch — attention pooling over full sequence
        x_embed          = self.embedding(rnn_input)
        x_lstm, _        = self.bilstm(x_embed)
        x_rnn_pooled     = self.attn_pool(x_lstm)       # (B, 512)
        x_rnn_proj       = self.rnn_proj(x_rnn_pooled)  # (B, 768)

        # Transformer branches — CLS token
        x1 = self.bert1(
            input_ids=input_ids_1, attention_mask=attention_mask_1
        ).last_hidden_state[:, 0, :]

        x2 = self.bert2(
            input_ids=input_ids_2, attention_mask=attention_mask_2
        ).last_hidden_state[:, 0, :]

        # Stack and fuse
        x_all  = torch.stack([x_rnn_proj, x1, x2], dim=1)  # (B, 3, 768)
        fused  = self.fusion(x_all)                          # (B, 768)
        fused  = self.dropout(fused)
        fused  = self.ffn(fused)
        logits = self.classifier(fused)
        return logits

# ------------------ 10. TRAINING SETUP ------------------
vocab_size = min(max_words, len(tokenizer_cnn.word_index) + 1)
model      = FusionModel(vocab_size=vocab_size, embed_dim=256, num_classes=num_classes).to(device)

optimizer  = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)

# FIX: Focal loss with label smoothing — better than plain CE for imbalance
criterion  = FocalLoss(weight=class_weights, gamma=2.0, label_smoothing=0.1)

epochs        = 20
total_steps   = len(train_loader) * epochs
warmup_steps  = 2 * len(train_loader)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler()

# ------------------ 11. TRAINING LOOP WITH EARLY STOPPING ------------------
best_f1         = 0.0
patience        = 4
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()
        with autocast():
            logits = model(
                batch['input_ids_1'].to(device),
                batch['attention_mask_1'].to(device),
                batch['input_ids_2'].to(device),
                batch['attention_mask_2'].to(device),
                batch['rnn_input'].to(device)
            )
            loss = criterion(logits, batch['label'].to(device))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    # ---- Evaluation on clean dev set ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dev_loader:
            with autocast():
                logits = model(
                    batch['input_ids_1'].to(device),
                    batch['attention_mask_1'].to(device),
                    batch['input_ids_2'].to(device),
                    batch['attention_mask_2'].to(device),
                    batch['rnn_input'].to(device)
                )
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch['label'].numpy())

    current_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss   = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Macro F1: {current_f1:.4f}")

    if current_f1 > best_f1:
        best_f1 = current_f1
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  ✓ New best F1: {best_f1:.4f} — model saved")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print("Early stopping triggered.")
            break

# ------------------ 12. FINAL EVALUATION ------------------
print("\nLoading best model for final evaluation on clean dev set...")
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in dev_loader:
        with autocast():
            logits = model(
                batch['input_ids_1'].to(device),
                batch['attention_mask_1'].to(device),
                batch['input_ids_2'].to(device),
                batch['attention_mask_2'].to(device),
                batch['rnn_input'].to(device)
            )
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

target_names = [
    'Not_offensive',
    'Offensive_Targeted_Insult_Individual',
    'Offensive_Targeted_Insult_Group',
    'Offensive_Untargetede',
    'not-malayalam'
]
print("\nFinal Classification Report (on original, non-augmented dev set):")
print(classification_report(all_labels, all_preds, target_names=target_names))
print(f"Best Macro F1: {best_f1:.4f}")

Train size after augmentation: 19390
Dev size (original only):      3602
Train class distribution: Counter({0: 12746, 4: 4640, 1: 840, 3: 676, 2: 488})
Dev class distribution:   Counter({0: 3186, 4: 290, 1: 53, 3: 42, 2: 31})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Epoch 19/20 | Loss: 0.4210 | Macro F1: 0.7164
  No improvement (3/4)


/tmp/ipykernel_2851/3144061688.py:358: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_2851/3144061688.py:381: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 20/20 | Loss: 0.4211 | Macro F1: 0.7164
  No improvement (4/4)
Early stopping triggered.

Loading best model for final evaluation on clean dev set...


/tmp/ipykernel_2851/3144061688.py:417: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Final Classification Report (on original, non-augmented dev set):
                                      precision    recall  f1-score   support

                       Not_offensive       0.97      0.98      0.98      3186
Offensive_Targeted_Insult_Individual       0.79      0.42      0.54        53
     Offensive_Targeted_Insult_Group       0.94      0.52      0.67        31
               Offensive_Untargetede       0.60      0.57      0.59        42
                       not-malayalam       0.85      0.90      0.87       290

                            accuracy                           0.96      3602
                           macro avg       0.83      0.68      0.73      3602
                        weighted avg       0.96      0.96      0.95      3602

Best Macro F1: 0.7291
